<a href="https://colab.research.google.com/github/ilaydacepniogluu-sys/AkademiQ_DataScience/blob/main/AkademiQHafta7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#DBSCAN (Density Based Spatial Clustering of Applications with Noise)




In [ ]:
#Veriyi yoğunluklarına göre kümelerine ayırır.
#Gürültüyü otomatik şekilde ayıklayabilen, Cluster sayısını önceden bilmek zorunda olmayan ve karmaşık şekilli kümeleri yakalayabilen bir algoritmadır.
#Müşteri davranış segmentasyonunda da kullanılır.
#NOT:burada bazı  noktalar bazen hiçbir yere ait olmayabiliyor.

eps ( epsilon) = Bir noktanın çevresindeki komşuluk yarıçapı

min_samples = Bir bölgenin "yoğun" kabul edilmesi için gereken minimum nokta sayısı

#Nokta türleri

Core Point = Yoğun bölgenin merkezi


Border point = yoğun bölgeye yakın ama yeterince yoğun olmayan bölge


Noise Point = Hiiçbir yoğunluğa ait olmayan nokta


In [1]:
import pandas as pd
import numpy as np

from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors

import plotly.express as px # Web uyumlu görselleştirme elde edebilmek (python tarafında)
import plotly.graph_objects as go  #Bunlar görselleştirme grafikleridir.

In [3]:
df = pd.read_excel("Online Retail.xlsx")

In [4]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[ns]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 33.1+ MB


In [6]:
df = df.dropna(subset=["CustomerID"])

df["TotalPrice"] = df["Quantity"] * df["UnitPrice"]


In [7]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 406829 entries, 0 to 541908
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    406829 non-null  object        
 1   StockCode    406829 non-null  object        
 2   Description  406829 non-null  object        
 3   Quantity     406829 non-null  int64         
 4   InvoiceDate  406829 non-null  datetime64[ns]
 5   UnitPrice    406829 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      406829 non-null  object        
 8   TotalPrice   406829 non-null  float64       
dtypes: datetime64[ns](1), float64(3), int64(1), object(4)
memory usage: 31.0+ MB


In [9]:
customer_df = df.groupby("CustomerID").agg({ #her müşteri için toplam sipariş,toplam ürün ve toplam harcama hesaplıyoruz.
   "InvoiceNo":"count",
   "Quantity":"sum",
   "TotalPrice":"sum"

}).reset_index()

customer_df.columns=[
    "CuatomerID","TotalOrders","TotalQuantity","TotalSpend"
]
customer_df.head(10)

,CuatomerID,TotalOrders,TotalQuantity,TotalSpend
0,12346.0,2,0,0.00
1,12347.0,182,2458,4310.00
2,12348.0,31,2341,1797.24
3,12349.0,73,631,1757.55
4,12350.0,17,197,334.40
5,12352.0,95,470,1545.41
6,12353.0,4,20,89.00
7,12354.0,58,530,1079.40
8,12355.0,13,240,459.40
9,12356.0,59,1591,2811.43


In [10]:
features = customer_df[["TotalOrders","TotalQuantity","TotalSpend"]]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(features)

In [11]:
scaled_df = pd.DataFrame(
    X_scaled, columns=features.columns
)

display(scaled_df.head())

,TotalOrders,TotalQuantity,TotalSpend
0,-0.391720,-0.240215,-0.231001
1,0.382657,0.285870,0.293432
2,-0.266959,0.260828,-0.012316
3,-0.086271,-0.105162,-0.017146
4,-0.327188,-0.198051,-0.190312


# K-Distance Plot ile Eps Bulma

In [12]:
neighbors = NearestNeighbors(n_neighbors=5) #Amacımız her noktanın en yakın komşularını bulabilmek.
#n_neighbors=5 : Sisteme Şunu söyleriz; Her nokta için en yakın 5 tane komşuyu bul.
neighbors_fit = neighbors.fit(X_scaled)
#neighbors_fit : Bunu öğren. ölçeklendirdiğimiz verinin uzaydaki konumlarını öğrenir.
distance, indices = neighbors_fit.kneighbors(X_scaled)
#Her noktanın komşularına olan uzaklığı.
distance = np.sort(distance[:,4])

fig = px.line(
    distance, title="K-Distance ile epsilon seçimi"
)

fig.show()

In [13]:
dbscan = DBSCAN(
    eps = 0.7 ,
    min_samples=5
)

clusters = dbscan.fit_predict(X_scaled)

customer_df["Cluster"] = clusters

#DBSCAN'de -1 etkiketi noise anlamına gelir. Fraud user, abnormal sensor, bot activity

In [14]:
display(customer_df.head())

,CuatomerID,TotalOrders,TotalQuantity,TotalSpend,Cluster
0,12346.0,2,0,0.00,0
1,12347.0,182,2458,4310.00,0
2,12348.0,31,2341,1797.24,0
3,12349.0,73,631,1757.55,0
4,12350.0,17,197,334.40,0


In [23]:
#Yukarıdaki cluster'da -1 görmek demek dbscan'ın hiçbir yoğunluğa ait olamadığı nokta anlamına gelir.

In [15]:
print(customer_df["Cluster"].value_counts())

Cluster
 0    4330
-1      42
Name: count, dtype: int64


In [16]:
noise_points = customer_df[
    customer_df["Cluster"]==-1
]
display(noise_points.head())

,CuatomerID,TotalOrders,TotalQuantity,TotalSpend,Cluster
55,12415.0,778,77242,123725.45,-1
330,12748.0,4642,24210,29072.10,-1
436,12901.0,125,20915,16293.10,-1
458,12931.0,102,23377,33462.81,-1
525,13027.0,26,17280,6912.00,-1


In [20]:
fig = px.scatter(
    customer_df,
    x="TotalSpend",
    y="TotalQuantity",
    color="Cluster",
    title="DBSCAN Customer Cluster"
)

fig.show()

In [21]:
fig = px.scatter_3d(
    customer_df,
    x = "TotalSpend",
    y = "TotalQuantity",
    z = "TotalOrders",
    color = "Cluster",

)

fig.show()



In [22]:
customer_df.groupby("Cluster").mean(numeric_only=True)


,CuatomerID,TotalOrders,TotalQuantity,TotalSpend
Cluster,,,,
-1,15144.119048,1166.928571,33430.309524,57413.067143
0,15301.186605,82.636952,808.964203,1359.980830


#PCA ile görselleştirme


In [23]:
pca = PCA(n_components=2)

X_pca =pca.fit_transform(X_scaled)


In [24]:
customer_df["PCA-1"] = X_pca[:,0]
customer_df["PCA-2"]=  X_pca[:,1]

In [25]:
display(customer_df[["PCA-1", "PCA-2","Cluster"]].head())

,PCA-1,PCA-2,Cluster
0,-0.475327,-0.196386,0
1,0.539155,0.153233,0
2,0.034853,-0.314594,0
3,-0.116477,-0.038248,0
4,-0.393735,-0.165722,0
